# Sprint 2 — Working with Text Data
### Projeto LLM — Experimentos do pipeline de pré-processamento de dados

Este notebook reproduz, em um ambiente Google Colab autocontido, os componentes e experimentos da Sprint 2 do Projeto LLM, baseados no Capítulo 2 (*Working with Text Data*) do livro *Build a Large Language Model (From Scratch)* (Sebastian Raschka, Manning, 2025).

Fluxo implementado:

```
Texto → Tokenização → Tokens → Token IDs → Sequências de treinamento
      → Embeddings → Positional Embeddings → Lote de dados → Entrada do modelo
```

**Autores:** Matheus Dapper, Vitória Aparecida Vendausen

> **Sobre o corpus:** este notebook usa *The Verdict* (Edith Wharton, domínio público), o corpus de exemplo do próprio capítulo. O corpus final do projeto (texto técnico de eletrônica em português, definido no README raiz do repositório) ainda será integrado; trocar o corpus não exige mudar nenhuma célula abaixo além da célula de download/leitura do texto.


## 0. Configuração do ambiente
O Google Colab já vem com `torch` pré-instalado. Instalamos apenas `tiktoken` (tokenizador BPE do GPT-2) e baixamos o corpus de exemplo do capítulo.

In [ ]:
!pip install -q tiktoken

import re
import time
import io
import math

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt

torch.manual_seed(123)
print('torch:', torch.__version__)
print('tiktoken:', tiktoken.__version__)

In [ ]:
# Corpus de referência do capítulo: "The Verdict" (Edith Wharton, domínio público),
# o mesmo texto de exemplo usado no livro.
!wget -q -O the-verdict.txt https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt

with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Numero de caracteres: {len(raw_text)}')
print(raw_text[:300])

## 1. Tokenização
### 1.1 Tokenização didática baseada em regex
Divide o texto em palavras e sinais de pontuação. Serve para entender o conceito de "token" antes de introduzir o algoritmo real (BPE).

In [ ]:
def split_into_tokens(text: str) -> list[str]:
    """Tokenizacao baseada em regex (Secao 2.2 do capitulo)."""
    pieces = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    return [piece.strip() for piece in pieces if piece.strip()]

amostras = [
    'Hello, do you like tea?',
    'In the sunlit terraces of the palace.',
]
for frase in amostras:
    print(f'{frase!r} -> {split_into_tokens(frase)}')

### 1.2 Tokenização Byte Pair Encoding (BPE) — o tokenizador real do GPT
Usa a biblioteca `tiktoken` com o vocabulário do GPT-2. Diferente da versão por regex, o BPE consegue representar **qualquer** string, inclusive palavras inventadas, quebrando-as em subpalavras.

In [ ]:
bpe = tiktoken.get_encoding('gpt2')

def bpe_tokens(text, allowed_special={'<|endoftext|>'}):
    ids = bpe.encode(text, allowed_special=allowed_special)
    return [bpe.decode([i]) for i in ids], ids

amostras_bpe = amostras + ['Akwirw ier']  # palavra inventada: forca quebra em subpalavras
for frase in amostras_bpe:
    tokens, ids = bpe_tokens(frase)
    print(f'{frase!r}')
    print(f'  tokens : {tokens}')
    print(f'  ids    : {ids}')
    print(f'  decode : {bpe.decode(ids)!r}')

## 2. Vocabulário e Token IDs
Construção do vocabulário a partir do corpus (tokens únicos, ordenados alfabeticamente) e demonstração da relação **Token ↔ Token ID ↔ Vocabulário**, incluindo os tokens especiais `<|unk|>` e `<|endoftext|>`.

In [ ]:
SPECIAL_TOKENS = ['<|unk|>', '<|endoftext|>']

def build_vocabulary(text, special_tokens=SPECIAL_TOKENS):
    tokens = split_into_tokens(text)
    unique_tokens = sorted(set(tokens))
    if special_tokens:
        unique_tokens.extend(special_tokens)
    return {token: idx for idx, token in enumerate(unique_tokens)}

class Vocabulary:
    def __init__(self, str_to_int):
        self.str_to_int = str_to_int
        self.int_to_str = {i: s for s, i in str_to_int.items()}

    def __len__(self):
        return len(self.str_to_int)

    def token_to_id(self, token):
        if token in self.str_to_int:
            return self.str_to_int[token]
        return self.str_to_int['<|unk|>']

    def id_to_token(self, token_id):
        return self.int_to_str[token_id]

    def encode(self, text):
        return [self.token_to_id(t) for t in split_into_tokens(text)]

    def decode(self, ids):
        return ' '.join(self.id_to_token(i) for i in ids)

vocab = Vocabulary(build_vocabulary(raw_text))
print(f'Tamanho do vocabulario proprio (com tokens especiais): {len(vocab)}')
print(f'Tamanho do vocabulario BPE do GPT-2: {bpe.n_vocab}')

frase = 'Hello, do you like tea?'
ids = vocab.encode(frase)
print(f'\ntexto      : {frase!r}')
print(f'token ids  : {ids}')
print(f'decodificado: {vocab.decode(ids)!r}  (note o uso de <|unk|>)')

## 3. Preparação das sequências de treinamento (janela deslizante + DataLoader)
A partir dos Token IDs do corpus, geramos pares (entrada, alvo) por janela deslizante: o alvo é a entrada deslocada em uma posição. É essa técnica que transforma o corpus em um dataset de treinamento auto-supervisionado, sem qualquer rotulação manual.

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128,
                          shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDatasetV1(text, bpe, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                       drop_last=drop_last, num_workers=num_workers)


dataloader = create_dataloader_v1(raw_text, batch_size=4, max_length=8, stride=8, shuffle=False)
inputs, targets = next(iter(dataloader))
print(f'Numero de amostras no dataset: {len(dataloader.dataset)}')
print(f'Forma do lote de entrada : {tuple(inputs.shape)}')
print(f'Forma do lote de alvo    : {tuple(targets.shape)}')
print(f'\nPrimeiro par (entrada, alvo):')
print(f'  entrada: {inputs[0].tolist()}')
print(f'  alvo   : {targets[0].tolist()}')

## 4. Embeddings e Positional Embeddings
Os Token IDs são convertidos em vetores densos (`nn.Embedding`). Como a camada de embedding por si só não diferencia a posição de um token na sequência, somamos um segundo vetor — o *positional embedding* — indexado pela posição.

In [ ]:
GPT2_VOCAB_SIZE = 50257

def make_token_embedding_layer(vocab_size, output_dim, seed=123):
    torch.manual_seed(seed)
    return nn.Embedding(vocab_size, output_dim)

def make_positional_embedding_layer(context_length, output_dim, seed=123):
    torch.manual_seed(seed)
    return nn.Embedding(context_length, output_dim)

def build_input_embeddings(token_ids, token_embedding, positional_embedding):
    batch_size, context_length = token_ids.shape
    tok_embeds = token_embedding(token_ids)
    positions = torch.arange(context_length)
    pos_embeds = positional_embedding(positions)
    return tok_embeds + pos_embeds


OUTPUT_DIM, CONTEXT_LENGTH = 256, 4
dl = create_dataloader_v1(raw_text, batch_size=8, max_length=CONTEXT_LENGTH, stride=CONTEXT_LENGTH, shuffle=False)
inputs, targets = next(iter(dl))

tok_emb_layer = make_token_embedding_layer(GPT2_VOCAB_SIZE, OUTPUT_DIM)
pos_emb_layer = make_positional_embedding_layer(CONTEXT_LENGTH, OUTPUT_DIM)

token_embeddings = tok_emb_layer(inputs)
input_embeddings = build_input_embeddings(inputs, tok_emb_layer, pos_emb_layer)

print(f'Token IDs de entrada, forma : {tuple(inputs.shape)}')
print(f'Token embeddings, forma    : {tuple(token_embeddings.shape)}')
print(f'Input embeddings, forma    : {tuple(input_embeddings.shape)}')

# evidencia de que a posicao importa
id_exemplo = inputs[0, 0].item()
vetor_puro = tok_emb_layer(torch.tensor([id_exemplo]))[0]
vetor_pos0 = vetor_puro + pos_emb_layer(torch.tensor([0]))[0]
vetor_pos2 = vetor_puro + pos_emb_layer(torch.tensor([2]))[0]
print(f'\nToken ID {id_exemplo}: vetor na posicao 0 == vetor na posicao 2 ? '
      f"{'NAO' if not torch.allclose(vetor_pos0, vetor_pos2) else 'SIM'}")

## 5. Experimentação
Investigamos o impacto de diferentes configurações sobre os dados e suas representações: tamanho do contexto, tamanho do lote, dimensão de embedding, sobreposição (stride) e diferentes textos de entrada.

### 5.1 Tamanho do contexto × quantidade de amostras

In [ ]:
context_sizes = [4, 8, 16, 32, 64, 128]
total_tokens = len(bpe.encode(raw_text))

ctx_rows = []
for ctx in context_sizes:
    dl = create_dataloader_v1(raw_text, batch_size=1, max_length=ctx, stride=ctx, shuffle=False)
    n_amostras = len(dl.dataset)
    ctx_rows.append((ctx, n_amostras))
    print(f'contexto={ctx:>4}  amostras={n_amostras:>5}  cobertura={ctx*n_amostras/total_tokens:.1%}')

xs, ys = zip(*ctx_rows)
plt.figure(figsize=(6,4))
plt.plot(xs, ys, marker='o')
plt.xlabel('Tamanho do contexto (max_length)')
plt.ylabel('Amostras geradas (stride = contexto)')
plt.title('Contexto x quantidade de amostras')
plt.grid(alpha=0.3)
plt.show()

### 5.2 Sobreposição entre janelas (stride) × quantidade de amostras

In [ ]:
ctx = 32
strides = [8, 16, 32, 64]
stride_rows = []
for stride in strides:
    dl = create_dataloader_v1(raw_text, batch_size=1, max_length=ctx, stride=stride, shuffle=False)
    n_amostras = len(dl.dataset)
    stride_rows.append((stride, n_amostras))
    print(f'stride={stride:>3}  sobreposicao={max(0, ctx-stride):>3}  amostras={n_amostras}')

xs, ys = zip(*stride_rows)
plt.figure(figsize=(6,4))
plt.bar([str(x) for x in xs], ys)
plt.xlabel('stride (contexto fixo = 32)')
plt.ylabel('Amostras geradas')
plt.title('Stride x quantidade de amostras')
plt.show()

### 5.3 Tamanho do lote × forma dos tensores

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32]
ctx = 32
for bs in batch_sizes:
    dl = create_dataloader_v1(raw_text, batch_size=bs, max_length=ctx, stride=ctx, shuffle=False, drop_last=True)
    inputs, targets = next(iter(dl))
    print(f'batch_size={bs:>3}  lotes/epoca={len(dl):>4}  '
          f'forma entrada={tuple(inputs.shape)}  forma alvo={tuple(targets.shape)}')

### 5.4 Dimensão do embedding × estruturas produzidas

In [ ]:
dims = [16, 50, 128, 256, 768]
ctx, bs = 8, 4
dl = create_dataloader_v1(raw_text, batch_size=bs, max_length=ctx, stride=ctx, shuffle=False)
inputs, _ = next(iter(dl))

dim_rows = []
for dim in dims:
    tok_layer = make_token_embedding_layer(GPT2_VOCAB_SIZE, dim)
    pos_layer = make_positional_embedding_layer(ctx, dim)
    t0 = time.perf_counter()
    input_emb = build_input_embeddings(inputs, tok_layer, pos_layer)
    dt_ms = (time.perf_counter() - t0) * 1000
    n_params_tok = GPT2_VOCAB_SIZE * dim
    dim_rows.append((dim, n_params_tok, dt_ms))
    print(f'output_dim={dim:>4}  forma={tuple(input_emb.shape)}  '
          f'params_tokens={n_params_tok:>12,}  tempo={dt_ms:.3f} ms')

xs = [r[0] for r in dim_rows]
params = [r[1] for r in dim_rows]
tempos = [r[2] for r in dim_rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,4))
ax1.plot(xs, params, marker='o', color='tab:blue')
ax1.set_xlabel('output_dim'); ax1.set_ylabel('parametros (tabela de tokens)')
ax1.set_title('Dimensao do embedding x parametros'); ax1.grid(alpha=0.3)

ax2.plot(xs, tempos, marker='o', color='tab:orange')
ax2.set_xlabel('output_dim'); ax2.set_ylabel('tempo (ms)')
ax2.set_title('Dimensao do embedding x tempo'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 5.5 Quantidade de tokens produzidos para diferentes textos

In [ ]:
textos = {
    'frase curta (PT)': 'O modelo de linguagem transforma texto em numeros.',
    'frase curta (EN)': 'Language models transform text into numbers.',
    'palavra rara/inventada': 'Akwirw ier supercalifragilisticexpialidocious.',
    'paragrafo tecnico': (
        'A tokenizacao converte uma sequencia de caracteres em uma sequencia de '
        'tokens; cada token e entao mapeado para um Token ID por meio do '
        'vocabulario, e cada Token ID e finalmente convertido em um vetor '
        'denso pela camada de embeddings.'
    ),
    'the-verdict.txt (corpus completo)': raw_text,
}

nomes, n_chars_list, n_tokens_list = [], [], []
for nome, texto in textos.items():
    n_chars = len(texto)
    n_tokens = len(bpe.encode(texto, allowed_special={'<|endoftext|>'}))
    nomes.append(nome); n_chars_list.append(n_chars); n_tokens_list.append(n_tokens)
    print(f'{nome:<35} chars={n_chars:>6}  tokens(BPE)={n_tokens:>6}  tokens/char={n_tokens/n_chars:.3f}')

plt.figure(figsize=(7,4))
plt.barh(nomes, n_tokens_list, color='tab:green')
plt.xlabel('Tokens (BPE/GPT-2)')
plt.title('Quantidade de tokens por texto')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

## 6. Síntese dos resultados
- O número de amostras de treinamento cai de forma aproximadamente inversa ao tamanho do contexto (contexto 4 → 1286 amostras; contexto 128 → 40 amostras), para `stride = contexto`.
- Reduzir o `stride` (aumentar a sobreposição entre janelas) aumenta o número de amostras à custa de redundância entre elas.
- O `batch_size` só afeta o primeiro eixo dos tensores de entrada/alvo — não altera o conteúdo tokenizado, apenas quantas sequências são processadas em paralelo.
- O número de parâmetros da tabela de token embeddings cresce linearmente com `output_dim` (de ~800 mil, com `output_dim=16`, a ~38,6 milhões, com `output_dim=768`, apenas para o vocabulário do GPT-2).
- O tokenizador BPE produz uma quantidade de tokens muito mais estável entre idiomas/formatos do que a tokenização por regex, e nunca precisa recorrer a `<|unk|>`, mesmo para palavras inventadas.

A análise técnica completa, respondendo às dez questões do enunciado da Sprint, está no documento `docs/Analise_Sprint2.pdf` do repositório do projeto.